In [5]:
import tensorflow as tf
import pickle
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Load saved model
model = tf.keras.models.load_model(
    "D:/codes/placements/DL/senComp/lstm_model.h5"
)

# Load tokenizer
with open("D:/codes/placements/DL/senComp/tokenizer.pkl", "rb") as f:
    tokenizer = pickle.load(f)

# Load max_len
with open("D:/codes/placements/DL/senComp/max_len.pkl", "rb") as f:
    max_len = pickle.load(f)

# Load original dataset
df = pd.read_csv(
    "D:/codes/placements/DL/senComp/qoute_dataset.csv"
)

quotes = df["quote"].dropna().astype(str)

# Apply original preprocessing
import string

quotes = quotes.str.lower()
translator = str.maketrans("", "", string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

# Generate sequences
sentences = tokenizer.texts_to_sequences(quotes)

X = []
y = []

for seq in sentences:
    for i in range(1, len(seq)):
        X.append(seq[:i])
        y.append(seq[i])

# Pad sequences
X_pad = pad_sequences(
    X,
    maxlen=max_len,
    padding="pre"
)

y = np.array(y)

# Evaluate saved model
# Convert integer labels to one-hot encoding
y_onehot = tf.keras.utils.to_categorical(
    y, num_classes=10000
)

# Evaluate saved model (no retraining)
loss, accuracy = model.evaluate(
    X_pad,
    y_onehot,
    batch_size=128,
    verbose=1
)

print("\n========== MODEL RESULTS ==========")
print("Total Quotes:", len(quotes))
print("Vocabulary Size:", len(tokenizer.word_index) + 1)
print("Total Training Sequences:", len(X_pad))
print("Maximum Sequence Length:", max_len)
print(f"Training Accuracy: {accuracy * 100:.2f}%")
print(f"Training Loss: {loss:.4f}")

667/667 ━━━━━━━━━━━━━━━━━━━━ 283s 422ms/step - accuracy: 0.6208 - loss: 3.2533

========== MODEL RESULTS ==========
Total Quotes: 3038
Vocabulary Size: 8979
Total Training Sequences: 85271
Maximum Sequence Length: 745
Training Accuracy: 62.08%
Training Loss: 3.2533
